# Unified Python Graphiti on Databricks serverless

This notebook starts the same importable Python Graphiti runtime used by the CLI and Databricks App. The runtime owns Databricks model routing and uses the generated PostGraph driver against PostgreSQL. Search queries and results are also recorded in a Delta table.

No Node CLI or Uvicorn server is required.

In [ ]:
%pip install --upgrade "typing_extensions>=4.15" "dbx-tools-graphiti"

In [ ]:
dbutils.widgets.text("catalog", "reggie_pierce_aws_catalog", "Catalog")
dbutils.widgets.text("schema", "ai_search", "Schema")
dbutils.widgets.text("database_url", "", "PostgreSQL URL or Lakebase target")
dbutils.widgets.text("group_id", "serverless-demo", "Graphiti group")
dbutils.widgets.text("chat_model", "databricks-gpt-5-nano", "Chat model")
dbutils.widgets.text("embedding_model", "databricks-gte-large-en", "Embedding model")
dbutils.widgets.text("query", "What does Graphiti remember about PostgreSQL persistence?", "Search query")

In [ ]:
import json
import logging
import os
import re
from datetime import datetime, timezone

from databricks.sdk import WorkspaceClient
from dbx_tools.graphiti.runtime import GraphitiRuntime
from graphiti_core.nodes import EpisodeType

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("graphiti-serverless")


def _identifier(value: str) -> str:
    """Validate a widget value before using it as a SQL identifier."""
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value):
        raise ValueError(f"Invalid SQL identifier: {value!r}")
    return value


catalog = _identifier(dbutils.widgets.get("catalog"))
schema = _identifier(dbutils.widgets.get("schema"))
database_url = dbutils.widgets.get("database_url")
group_id = dbutils.widgets.get("group_id")
chat_model = dbutils.widgets.get("chat_model")
embedding_model = dbutils.widgets.get("embedding_model")
query = dbutils.widgets.get("query")

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS `{catalog}`.`{schema}`.`graphiti_query_history` (
      queried_at TIMESTAMP,
      group_id STRING,
      query STRING,
      result_json STRING
    ) USING DELTA
    """
)

workspace = WorkspaceClient()
auth_headers = workspace.config.authenticate()
authorization = next(
    value for key, value in auth_headers.items() if key.lower() == "authorization"
)
os.environ["DATABRICKS_HOST"] = workspace.config.host
os.environ["DATABRICKS_TOKEN"] = authorization.removeprefix("Bearer ")
os.environ["DATABRICKS_AUTH_TYPE"] = "pat"

In [ ]:
runtime_options = {
    "model": chat_model,
    "embedderModel": embedding_model,
}
if database_url:
    runtime_options["databaseUrl"] = database_url

runtime = GraphitiRuntime.from_options(runtime_options)
graphiti = await runtime.start()
logger.info("Unified PostGraph runtime started with %s and %s", chat_model, embedding_model)

In [ ]:
episode_time = datetime.now(timezone.utc)
episode = await graphiti.add_episode(
    name=f"serverless-persistence-{episode_time.isoformat()}",
    episode_body=(
        "Graphiti is running through the unified Python runtime in a Databricks "
        "serverless notebook. PostGraph stores graph memory in PostgreSQL while "
        "search history is saved in Delta."
    ),
    source=EpisodeType.text,
    source_description="Databricks serverless Graphiti persistence test",
    reference_time=episode_time,
    group_id=group_id,
)
logger.info(
    "Added episode %s with %d nodes and %d edges",
    episode.episode.uuid,
    len(episode.nodes),
    len(episode.edges),
)

In [ ]:
results = await graphiti.search(query, group_ids=[group_id])
result_json = json.dumps(
    [result.model_dump(mode="json") for result in results],
    default=str,
)
query_rows = [(datetime.now(timezone.utc), group_id, query, result_json)]
query_schema = "queried_at timestamp, group_id string, query string, result_json string"
(
    spark.createDataFrame(query_rows, query_schema)
    .write.mode("append")
    .saveAsTable(f"{catalog}.{schema}.graphiti_query_history")
)
logger.info("Saved query with %d results", len(results))
display(
    spark.sql(
        f"""
        SELECT queried_at, group_id, query, result_json
        FROM `{catalog}`.`{schema}`.`graphiti_query_history`
        WHERE group_id = '{group_id.replace("'", "''")}'
        ORDER BY queried_at DESC
        LIMIT 20
        """
    )
)

In [ ]:
await runtime.close()
logger.info("Unified Graphiti runtime closed and durable snapshot completed")